# Week 5, команда «Кудасики»: финальное решение с обучением не дольше 45 минут

Ноутбук **заново обучает** финальный ансамбль из четырёх сетей на всех размеченных снимках (train + val,
5698 штук) и пишет `final_submission.csv`. Время обучения измеряется `time.time()` и печатается крупно в
разделе 4. Лимит курса — 45 минут; мы целимся в 30–35 минут, чтобы был запас.

| GPU | модели, по очереди | вход, px | тест (TTA ×8), px | минут на Colab T4 (20 эпох) |
|---|---|---|---|---|
| 0 | ResNet-152 → ResNet-50 | 256 | 320 | 21.4 + 10.0 = 31.4 |
| 1 | ResNet-101 → Inception v3 | 256 / 299 | 320 / 363 | 15.2 + 12.8 = 28.1 |

Это **тот же ансамбль**, что дал 0.96678 на публичном лидерборде (val 0.9785, 15 ошибок из 699). Раньше модели
обучались по очереди на одной T4 — 59 минут. Здесь две видеокарты работают **одновременно**, каждая
обучает две модели, поэтому общее время — время более загруженной GPU, около 31–33 минут.

**Как запустить на Kaggle.**
1. *Add Input* → данные соревнования `week-5-competition-geo-satellites-classification`.
2. *Settings → Accelerator → GPU T4 x2*. Не P100: установленная версия torch собрана без поддержки P100
   (раздел 2 это проверяет).
3. *Settings → Internet → On*: нужен, чтобы скачать веса ImageNet из torchvision. Требует подтверждённого
   телефона в аккаунте Kaggle.
4. *Save Version → Save & Run All (Commit)*. Ноутбук выполнится в фоне, лог с временем сохранится в версии,
   а `final_submission.csv` появится в *Output*.

## Почему решение устроено именно так

* **Модели и рецепт не меняли.** Всё проверено в holdout-прогонах (`Team_final.ipynb`): ResNet-50/101/152
  (`IMAGENET1K_V2`) и Inception v3, обучение всей сети, AdamW, разогрев + косинус, label smoothing,
  аугментация искажениями `RandomCorruption`, 20 эпох. Размер тестового снимка (320 / 363 px) выбран на val
  ещё тогда и здесь зафиксирован.
* **Почему 4 модели, а не больше или меньше.** Ансамбль из 2 моделей (r50 + r101) дал 0.96506, из 4 — 0.96678,
  из 7 — тоже 0.96678. Модели ошибаются почти на одних и тех же снимках, поэтому после 4 моделей выигрыша
  нет, а время растёт.
* **Почему два процесса, а не `DataParallel`.** Каждая GPU обучает свои модели независимо: рецепт не
  меняется (тот же батч 32, та же BatchNorm), ускорение почти ровно ×2. `DataParallel` делит один батч
  на две GPU и упирается в один процесс Python: ускорение заметно меньше ×2, а BatchNorm видит по 16 снимков.
* **Ускорения без изменения рецепта:** `cudnn.benchmark` (cuDNN подбирает самые быстрые свёртки), `AdamW(fused=True)`,
  счётчики loss / accuracy на GPU без синхронизации на каждом шаге.
* **Предохранитель времени.** Первая эпоха — разогрев lr, и её ход не зависит от общего числа эпох. После неё
  известна реальная скорость этой GPU. Если 20 эпох всех моделей этой GPU не успевают к `BUDGET_MIN` минутам,
  число эпох уменьшается, и косинус строится уже на новое число эпох. При скорости наших прогонов на Colab
  предохранитель не срабатывает: в логе будет «20/20».
* **Test только для предсказаний.** Снимки test читаются в разделе 3, но процессы обучения открывают их
  только после конца обучения, чтобы предсказать. Никаких статистик по test.

## 1. Настройки
* `PLAN_2GPU` — какие модели обучает каждая GPU. Самая долгая модель идёт первой, тогда предохранитель
  времени сразу видит основную нагрузку.
* `PLAN_1GPU` — запасной план, если доступна только одна GPU: 3 модели по 16 эпох (около 31 минуты на T4).
* `BUDGET_MIN` — к этой минуте от старта таймера обучение должно закончиться (запас до 45 минут).
* `TTA_VIEWS` — сколько вариантов каждого тестового снимка усреднять: 8 = все повороты на 90° и отражения
  (так выбирали на val). Если преподаватели требуют предсказание test за 2 минуты (правило из исходного
  CPU-регламента), поставьте 2: предсказание займёт около 2 минут вместо 7, точность — чуть ниже (наша оценка).
* `QUICK_TEST = True` — проверка, что всё работает (1 эпоха, 260 снимков, 256 снимков test, 3–5 минут).
  Результат пишется в `final_submission_quick.csv`, а не в финальный файл.

In [ ]:
import time
NOTEBOOK_T0 = time.time()                     # для общего времени ноутбука
QUICK_TEST = False

PLAN_2GPU = {0: ["r152", "r50"], 1: ["r101", "inc3"]}
PLAN_1GPU = {0: ["r50", "inc3", "r101"]}
EPOCHS_2GPU, EPOCHS_1GPU = 20, 16
BUDGET_MIN = 40          # обучение должно закончиться к 40-й минуте от старта таймера (лимит 45)
TTA_VIEWS = 8            # 8 / 4 / 2 / 1
CUDNN_BENCHMARK = True   # False: детерминированные алгоритмы cuDNN (медленнее)
SEED = 43                # seed для Python, NumPy и torch задаётся в worker.py перед каждой моделью

# val accuracy holdout-моделей (обучены на train 20 эпох, проверены на val с TTA x8; Team_final.ipynb) — для таблицы
HOLDOUT_VAL = {"r50": 0.9742, "r101": 0.9700, "r152": 0.9714, "inc3": 0.9714, "inc3_363": 0.9771}
HOLDOUT_ENSEMBLE_VAL = {frozenset(["r50", "r101", "r152", "inc3"]): 0.9785,      # 15 ошибок из 699
                        frozenset(["r50", "r101", "inc3"]): 0.9728,              # 19 ошибок
                        frozenset(["r50", "inc3"]): 0.9757,                      # 17 ошибок
                        frozenset(["r50", "r152", "inc3", "inc3_363"]): 0.9814}  # 13 ошибок

## 2. Окружение
Печатаем, на чём идёт обучение: число ядер CPU, память, видеокарты, версии библиотек. P100 не подходит:
torch в образе Kaggle собран без кода для этой видеокарты (ошибка «no kernel image is available»).

In [ ]:
import json
import os
import platform
import socket
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import PIL
from PIL import Image
import torch
import torchvision

n_gpu = torch.cuda.device_count()
assert n_gpu >= 1, "нет GPU: Settings -> Accelerator -> GPU T4 x2"
gpus = [torch.cuda.get_device_name(i) for i in range(n_gpu)]
caps = [torch.cuda.get_device_capability(i) for i in range(n_gpu)]
if min(caps) < (7, 0):
    raise RuntimeError(f"{gpus}: эта версия torch не поддерживает P100 (sm_60). Выберите GPU T4 x2.")
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "pillow": PIL.__version__,
            "gpu": gpus, "cpu_count": os.cpu_count(), "ram_gb": round(ram_gb, 1)}
print(json.dumps(VERSIONS, ensure_ascii=False))
try:
    socket.create_connection(("download.pytorch.org", 443), timeout=5).close()
    print("интернет есть: веса ImageNet скачаются")
except OSError:
    print("!!! НЕТ ИНТЕРНЕТА: Settings -> Internet -> On (иначе веса ImageNet не скачать)")

PLAN = PLAN_2GPU if n_gpu >= 2 else PLAN_1GPU
EPOCHS = 1 if QUICK_TEST else (EPOCHS_2GPU if n_gpu >= 2 else EPOCHS_1GPU)
WORKERS_PER_GPU = max(1, os.cpu_count() // len(PLAN)) if len(PLAN) > 1 else max(2, os.cpu_count() - 1)
ALL_MODELS = [m for models_on_gpu in PLAN.values() for m in models_on_gpu]
print(f"план: {PLAN}, эпох: {EPOCHS}, процессов аугментации на GPU: {WORKERS_PER_GPU}")

## 3. Код обучения одной GPU (`worker.py`) и подготовка test
Ячейка ниже записывает `worker.py` — тот же код, что в `Team_final.ipynb` (аугментации, модели, цикл
обучения, TTA), плюс предохранитель времени. Каждая GPU запускает его отдельным процессом со своим списком
моделей.

In [ ]:
%%writefile worker.py
"""Обучение части ансамбля на ОДНОЙ видеокарте + предсказание test.

Ноутбук Kaggle_45min.ipynb запускает этот файл отдельным процессом на каждую GPU:
    CUDA_VISIBLE_DEVICES=0 python worker.py --gpu 0 --models r152 r50  --t0 <таймер ноутбука> ...
    CUDA_VISIBLE_DEVICES=1 python worker.py --gpu 1 --models r101 inc3 --t0 <таймер ноутбука> ...
Процессы ничего не обменивают во время обучения: каждый обучает свои модели на всех 5698 размеченных снимках
(train + val) по рецепту из Team_final.ipynb и сохраняет вероятности для test. Ансамбль собирает ноутбук.

Отличия от Team_final.ipynb (рецепт обучения тот же):
  * только этап full (обучение на train + val), без holdout: оценка на val и выбор размера TTA уже сделаны;
  * cudnn.benchmark = True: cuDNN сам выбирает самые быстрые свёртки (вход всегда одного размера);
  * AdamW(fused=True) и счётчики loss / accuracy на GPU: нет синхронизации CPU-GPU на каждом шаге;
  * «предохранитель» времени (time_guard): после первой эпохи (разогрев) известна реальная скорость; если
    20 эпох не успевают к BUDGET минут от старта таймера, число эпох уменьшается (косинус строится на новое
    число эпох). При скорости как в наших прогонах на Colab он не срабатывает.
"""
import argparse
import gc
import io
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

parser = argparse.ArgumentParser()
parser.add_argument("--gpu", required=True, help="номер GPU (для подписей; саму GPU задаёт CUDA_VISIBLE_DEVICES)")
parser.add_argument("--models", nargs="+", required=True, help="ключи из PRESETS, в порядке обучения")
parser.add_argument("--t0", type=float, required=True, help="time.time() в ноутбуке в момент старта таймера")
parser.add_argument("--budget", type=float, default=40.0, help="к этой минуте от t0 обучение должно закончиться")
parser.add_argument("--epochs", type=int, default=20)
parser.add_argument("--min-epochs", type=int, default=10)
parser.add_argument("--tta", type=int, default=8, choices=[1, 2, 4, 8], help="число вариантов снимка в TTA")
parser.add_argument("--workers", type=int, default=2, help="процессы DataLoader (аугментации на CPU)")
parser.add_argument("--benchmark", type=int, default=1, help="1: cudnn.benchmark; 0: детерминированный cuDNN")
parser.add_argument("--arrays", required=True, help="папка с trainval_images.npy, trainval_labels.npy, test_images.npy")
parser.add_argument("--out", required=True, help="папка для результатов")
args = parser.parse_args()

T0, BUDGET, OUT, ARRAYS = args.t0, args.budget, Path(args.out), Path(args.arrays)
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda"
NUM_CLASSES = 20
torch.set_num_threads(1)      # главный процесс почти не считает на CPU; ядра CPU нужны процессам аугментации


def log(message):
    print(f"[GPU {args.gpu} | {(time.time() - T0) / 60:5.1f} мин] {message}", flush=True)


# ================================================================================ настройки (как в Team_final)
COMMON = dict(lr=2e-4, head_lr_mult=10, weight_decay=0.01, label_smoothing=0.1, warmup_epochs=1,
              crop_scale=(0.5, 1.0), corruption_p=0.5, jitter=0.3, dropout=0.2, epochs=args.epochs, batch=32,
              seed=43)
PRESETS = {
    # test_sizes — размер тестового снимка, выбранный на val в holdout-прогонах (раздел 9 Team_final.ipynb)
    "r50": dict(arch="resnet50", size=256, test_sizes=[320]),
    "r101": dict(arch="resnet101", size=256, test_sizes=[320]),
    "r152": dict(arch="resnet152", size=256, test_sizes=[320]),
    "inc3": dict(arch="inception_v3", size=299, test_sizes=[363]),
    "inc3_363": dict(arch="inception_v3", size=363, test_sizes=[427]),
}
# минут на эпоху full-обучения (5698 снимков) в наших прогонах на Colab T4 — для прогноза времени
COLAB_EPOCH_MIN = {"r50": 0.502, "r101": 0.759, "r152": 1.067, "inc3": 0.642, "inc3_363": 0.922}


def get_cfg(name):
    return {**COMMON, **PRESETS[name]}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = bool(args.benchmark)
    torch.backends.cudnn.deterministic = not args.benchmark


# ================================================================================ аугментации (как в Team_final)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
GRAY_WEIGHTS = torch.tensor([0.299, 0.587, 0.114], device=DEVICE).view(1, 3, 1, 1)


class RandomCorruption:
    # С вероятностью p применяет 1 или 2 случайных искажения, похожих на найденные в val-снимках.
    OPS = ("blur", "low_res", "jpeg", "gray", "noise", "low_contrast")

    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img):
        if random.random() >= self.p:
            return img
        for op in random.sample(self.OPS, k=random.choice([1, 1, 2])):
            img = getattr(self, op)(img)
        return img

    def blur(self, img):                     # гауссово размытие, sigma 0.8...3 px
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 3.0)))

    def low_res(self, img):                  # уменьшаем в 2...6 раз и увеличиваем обратно
        size = img.width
        small = max(8, int(size * random.uniform(1 / 6, 1 / 2)))
        return img.resize((small, small), Image.BILINEAR).resize((size, size), Image.BILINEAR)

    def jpeg(self, img):                     # сильное JPEG-сжатие, качество 8...40
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=random.randint(8, 40))
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")

    def gray(self, img):                     # оттенки серого, 3 канала сохраняются
        return img.convert("L").convert("RGB")

    def noise(self, img):                    # гауссов шум, std 5...20 уровней яркости
        rng = np.random.default_rng(random.getrandbits(32))
        pixels = np.asarray(img, dtype=np.float32)
        pixels = pixels + rng.normal(0, random.uniform(5, 20), pixels.shape)
        return Image.fromarray(np.clip(pixels, 0, 255).astype(np.uint8))

    def low_contrast(self, img):             # контраст x0.4...0.8
        return ImageEnhance.Contrast(img).enhance(random.uniform(0.4, 0.8))


class RandomRot90:
    ROTATIONS = (None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270)

    def __call__(self, img):
        rotation = random.choice(self.ROTATIONS)
        return img if rotation is None else img.transpose(rotation)


class ToUint8Tensor:
    def __call__(self, img):
        return torch.from_numpy(np.array(img, dtype=np.uint8)).permute(2, 0, 1)


def make_train_transform(size, cfg):
    return transforms.Compose([
        RandomCorruption(p=cfg["corruption_p"]),
        transforms.RandomResizedCrop(size, scale=cfg["crop_scale"]),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        RandomRot90(),
        ToUint8Tensor(),
    ])


def gpu_color_jitter(x, strength, generator):
    def factor():
        return 1 + (torch.rand(x.shape[0], 1, 1, 1, device=x.device, generator=generator) * 2 - 1) * strength
    x = x * factor()                                                      # яркость
    mean = (x * GRAY_WEIGHTS).sum(1, keepdim=True).mean(dim=(2, 3), keepdim=True)
    x = (x - mean) * factor() + mean                                      # контраст
    gray = (x * GRAY_WEIGHTS).sum(1, keepdim=True)
    x = (x - gray) * factor() + gray                                      # насыщенность
    return x.clamp(0, 1)


def to_model_input(x_u8, size=None, jitter=0.0, generator=None):
    x = x_u8.to(DEVICE, non_blocking=True).float().div_(255)
    if jitter > 0:
        x = gpu_color_jitter(x, jitter, generator)
    if size is not None and x.shape[-1] != size:
        x = F.interpolate(x, size=(size, size), mode="bilinear", antialias=True, align_corners=False)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x.contiguous(memory_format=torch.channels_last)


class ArrayDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        return self.transform(Image.fromarray(np.array(self.images[i]))), int(self.labels[i])


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_train_loader(images, labels, cfg):
    generator = torch.Generator().manual_seed(cfg["seed"])
    return DataLoader(ArrayDataset(images, labels, make_train_transform(cfg["size"], cfg)),
                      batch_size=cfg["batch"], shuffle=True, drop_last=True, num_workers=args.workers,
                      worker_init_fn=seed_worker, generator=generator, pin_memory=True,
                      persistent_workers=args.workers > 0,
                      # fork (как на Colab): процессы аугментации получают массив снимков без копирования
                      multiprocessing_context="fork" if args.workers > 0 else None)


# ================================================================================ модели (как в Team_final)
BACKBONES = {
    "resnet50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V2),
    "resnet101": (models.resnet101, models.ResNet101_Weights.IMAGENET1K_V2),
    "resnet152": (models.resnet152, models.ResNet152_Weights.IMAGENET1K_V2),
    "inception_v3": (models.inception_v3, models.Inception_V3_Weights.IMAGENET1K_V1),
}


def build_model(cfg):
    constructor, weights = BACKBONES[cfg["arch"]]
    body = constructor(weights=weights)      # веса уже скачаны ноутбуком в кэш torchvision
    if cfg["arch"] == "inception_v3":        # transform_input=True ставят сами веса IMAGENET1K_V1
        body.aux_logits, body.AuxLogits = False, None
        body.dropout = nn.Identity()
    n_features = body.fc.in_features
    body.fc = nn.Identity()
    head = nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(n_features, NUM_CLASSES))
    return nn.Sequential(body, head)


# ================================================================================ обучение
def lr_factor(step, warmup_steps, total_steps):
    # разогрев: lr линейно растёт первую эпоху; затем косинус до 0 к последнему шагу
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / max(1, total_steps - warmup_steps)))


def time_guard(name, epoch1_min, epochs, later):
    """После первой эпохи: успеваем ли все оставшиеся эпохи этой модели и следующих моделей к BUDGET минуте?
    Если нет — уменьшаем число эпох этой модели (следующие проверяются так же после своей первой эпохи)."""
    now = (time.time() - T0) / 60
    speed = epoch1_min / COLAB_EPOCH_MIN[name]                       # >1 — медленнее, чем на Colab
    rest_this = (epochs - 1) * epoch1_min
    rest_later = speed * sum(COLAB_EPOCH_MIN[m] * args.epochs for m in later)
    projected = now + rest_this + rest_later
    log(f"[guard] {name}: 1-я эпоха {epoch1_min:.2f} мин ({speed:.2f}x от Colab); прогноз конца обучения на этой GPU "
        f"{projected:.1f} мин (бюджет {BUDGET:.0f})")
    if projected <= BUDGET:
        return epochs
    share = max(0.0, (BUDGET - now) / (rest_this + rest_later))
    new_epochs = max(args.min_epochs, 1 + int(share * (epochs - 1)))
    log(f"[guard] !!! {name}: не успеваем — эпох {epochs} -> {new_epochs}")
    return new_epochs


def train_model(name, images, labels, later):
    cfg = get_cfg(name)
    seed_everything(cfg["seed"])
    model = build_model(cfg).to(DEVICE, memory_format=torch.channels_last)
    loader = make_train_loader(images, labels, cfg)
    steps_per_epoch = len(loader)
    epochs = cfg["epochs"]
    optimizer = torch.optim.AdamW([
        {"params": model[0].parameters(), "lr": cfg["lr"]},
        {"params": model[1].parameters(), "lr": cfg["lr"] * cfg["head_lr_mult"]},
    ], weight_decay=cfg["weight_decay"], fused=True)
    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = cfg["warmup_epochs"] * steps_per_epoch
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    scaler = torch.amp.GradScaler("cuda")
    jitter_generator = torch.Generator(device=DEVICE).manual_seed(cfg["seed"])

    history, step, epoch = [], 0, 0
    while epoch < epochs:                    # epochs может уменьшиться после 1-й эпохи (time_guard)
        epoch += 1
        model.train()
        start = time.time()
        total_steps = epochs * steps_per_epoch
        loss_sum = torch.zeros((), device=DEVICE)
        correct = torch.zeros((), device=DEVICE, dtype=torch.long)
        seen = 0
        for x_u8, y in loader:
            factor = lr_factor(step, warmup_steps, total_steps)
            for group, base_lr in zip(optimizer.param_groups, base_lrs):
                group["lr"] = base_lr * factor
            x = to_model_input(x_u8, jitter=cfg["jitter"], generator=jitter_generator)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(x)
                loss = criterion(outputs, y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.detach().float() * len(y)       # счётчики остаются на GPU: без синхронизации
            correct += (outputs.argmax(1) == y).sum()
            seen += len(y)
            step += 1
        torch.cuda.synchronize()
        minutes = (time.time() - start) / 60
        history.append({"epoch": epoch, "train_loss": loss_sum.item() / seen, "train_acc": correct.item() / seen,
                        "lr_factor": factor, "minutes": minutes})
        log(f"{name} эпоха {epoch}/{epochs}: loss {history[-1]['train_loss']:.4f}, "
            f"acc {history[-1]['train_acc']:.4f}, {minutes:.2f} мин")
        if epoch == 1:
            epochs = time_guard(name, minutes, epochs, later)
    return model, pd.DataFrame(history), epochs


# ================================================================================ предсказание
# варианты снимка для TTA: (число поворотов на 90°, отражение). 8 = все повороты и отражения (группа D4)
VIEW_SETS = {8: [(k, flip) for flip in (False, True) for k in range(4)],
             4: [(0, False), (2, False), (0, True), (2, True)],
             2: [(0, False), (0, True)],
             1: [(0, False)]}


@torch.no_grad()
def predict_proba(model, images, size, views, batch=128):
    model.eval()
    result = []
    for start in range(0, len(images), batch):
        x = to_model_input(torch.from_numpy(np.array(images[start:start + batch])).permute(0, 3, 1, 2), size)
        probs = 0
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            for k, flip in views:
                v = x.flip(3) if flip else x
                v = torch.rot90(v, k, dims=(2, 3)).contiguous(memory_format=torch.channels_last)
                probs = probs + model(v).float().softmax(1)
        result.append((probs / len(views)).cpu())
    return torch.cat(result).numpy()


# ================================================================================ main
def save_timing(timing):
    # сначала во временный файл, потом переименование: ноутбук никогда не прочитает файл наполовину
    tmp = OUT / f"timing_gpu{args.gpu}.json.tmp"
    tmp.write_text(json.dumps(timing, indent=1, ensure_ascii=False), encoding="utf-8")
    tmp.replace(OUT / f"timing_gpu{args.gpu}.json")


def main():
    log(f"{torch.cuda.get_device_name(0)} | torch {torch.__version__} | модели {args.models} | "
        f"DataLoader workers {args.workers} | cudnn.benchmark {bool(args.benchmark)}")
    images = np.load(ARRAYS / "trainval_images.npy", mmap_mode="r")      # общий файл, в память не копируется
    labels = np.load(ARRAYS / "trainval_labels.npy")
    timing = {"gpu": args.gpu, "models": args.models, "t0": T0, "t_worker_start": time.time(), "per_model": {}}

    # 1) обучение всех моделей этой GPU; test в это время не трогаем
    trained = {}
    for i, name in enumerate(args.models):
        start = time.time()
        model, history, epochs_used = train_model(name, images, labels, later=args.models[i + 1:])
        history.to_csv(OUT / f"history_{name}.csv", index=False)
        timing["per_model"][name] = {"train_min": (time.time() - start) / 60, "epochs": epochs_used}
        trained[name] = model.cpu()              # обученная модель ждёт предсказания в памяти CPU
        del model
        gc.collect()
        torch.cuda.empty_cache()
        log(f"{name} обучена за {timing['per_model'][name]['train_min']:.1f} мин ({epochs_used} эпох)")
    timing["t_train_end"] = time.time()
    save_timing(timing)                          # ноутбук видит, что обучение на этой GPU закончено
    log(f"ОБУЧЕНИЕ НА ЭТОЙ GPU ЗАКОНЧЕНО: {(timing['t_train_end'] - T0) / 60:.1f} мин от старта таймера")

    # 2) предсказание test обученными моделями (test используется только здесь)
    test_images = np.load(ARRAYS / "test_images.npy", mmap_mode="r")
    for name in args.models:
        start = time.time()
        model = trained.pop(name).to(DEVICE, memory_format=torch.channels_last)
        sizes = get_cfg(name)["test_sizes"]
        probs = np.mean([predict_proba(model, test_images, s, VIEW_SETS[args.tta]) for s in sizes], axis=0)
        np.save(OUT / f"test_probs_{name}.npy", probs.astype(np.float32))
        timing["per_model"][name]["predict_min"] = (time.time() - start) / 60
        log(f"{name}: test предсказан ({sizes} px, TTA x{args.tta}) за "
            f"{timing['per_model'][name]['predict_min']:.1f} мин")
        del model
        torch.cuda.empty_cache()
    timing["t_pred_end"] = time.time()
    save_timing(timing)
    log("готово")


if __name__ == "__main__":
    main()

**Данные.** Ищем файлы соревнования в `/kaggle/input` (или в текущей папке, если запуск не на Kaggle).
Снимки **test** читаются здесь, до таймера обучения: это подготовка к предсказанию, а не обучение. Они
сохраняются в общий файл, который процессы обучения откроют только после конца обучения.

In [ ]:
def find_data_dir():
    found = [p.parent for p in Path("/kaggle/input").rglob("train.csv")] if Path("/kaggle/input").exists() else []
    found += [p for p in (Path("."), Path("data")) if (p / "train.csv").exists()]
    for p in found:
        if (p / "train").is_dir() and (p / "test").is_dir() and (p / "test.csv").exists():
            return p
    raise FileNotFoundError("не нашли train.csv / train / test: добавьте данные соревнования (Add Input)")

DATA_DIR = find_data_dir()
ARRAYS = Path("/tmp/lab5_arrays"); ARRAYS.mkdir(parents=True, exist_ok=True)   # большие массивы — не в Output
OUT = Path("runs45"); OUT.mkdir(exist_ok=True)

train_table = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
val_table = pd.read_csv(DATA_DIR / "val.csv", dtype={"id": str})
test_table = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
classes = pd.read_csv(DATA_DIR / "classes.csv").sort_values("label").class_name.tolist()
if QUICK_TEST:   # по 10 train- и 3 val-снимка на класс, первые 256 снимков test
    train_table = train_table.groupby("label").head(10).reset_index(drop=True)
    val_table = val_table.groupby("label").head(3).reset_index(drop=True)
    test_table = test_table.head(256).reset_index(drop=True)

def read_image(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        if img.size != (256, 256):
            img = img.resize((256, 256), Image.BICUBIC)
        return np.asarray(img)

def load_images(table, folder):
    # снимки -> массив uint8 (N, 256, 256, 3); JPEG декодируются в несколько потоков
    paths = [DATA_DIR / folder / (i if i.endswith(".jpg") else f"{i}.jpg") for i in table.id]
    images = np.empty((len(paths), 256, 256, 3), dtype=np.uint8)
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
        for k, array in enumerate(pool.map(read_image, paths)):
            images[k] = array
    return images

start = time.time()
np.save(ARRAYS / "test_images.npy", load_images(test_table, "test"))
print(f"данные: {DATA_DIR} | train {len(train_table)}, val {len(val_table)}, test {len(test_table)}, "
      f"классов {len(classes)} | test прочитан за {(time.time() - start) / 60:.1f} мин")

## 4. Обучение (таймер)
**Что внутри таймера:** чтение train + val с диска, запуск процессов, скачивание весов ImageNet, все эпохи
всех моделей на обеих GPU. **Время обучения** = момент, когда закончила **последняя** GPU, минус старт
таймера. Это обычное время по часам (wall-clock): GPU работают одновременно, поэтому их минуты не
складываются.

**Что не входит:** чтение test (раздел 3) и предсказание test, его время печатается отдельно.

Раз в минуту печатается последняя строка лога каждой GPU; полные логи — `runs45/log_gpu*.txt`.

In [ ]:
T0 = time.time()                                   # ======== СТАРТ ТАЙМЕРА ОБУЧЕНИЯ ========
trainval_table = pd.concat([train_table, val_table], ignore_index=True)
np.save(ARRAYS / "trainval_images.npy", np.concatenate([load_images(train_table, "train"),
                                                         load_images(val_table, "val")]))
np.save(ARRAYS / "trainval_labels.npy", trainval_table.label.to_numpy())
print(f"train + val прочитаны: {len(trainval_table)} снимков, {(time.time() - T0) / 60:.1f} мин от старта")

procs, logs = {}, {}
for gpu, models_on_gpu in PLAN.items():
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    cmd = [sys.executable, "worker.py", "--gpu", str(gpu), "--models", *models_on_gpu, "--t0", repr(T0),
           "--budget", str(BUDGET_MIN), "--epochs", str(EPOCHS), "--tta", str(TTA_VIEWS),
           "--workers", str(WORKERS_PER_GPU), "--benchmark", str(int(CUDNN_BENCHMARK)),
           "--arrays", str(ARRAYS), "--out", str(OUT)]
    logs[gpu] = OUT / f"log_gpu{gpu}.txt"
    procs[gpu] = subprocess.Popen(cmd, env=env, stdout=open(logs[gpu], "w"), stderr=subprocess.STDOUT)
    print(f"GPU {gpu}: запущено обучение {models_on_gpu}")

def last_line(path):
    lines = [l for l in path.read_text(encoding="utf-8", errors="replace").splitlines() if l.strip()]
    return lines[-1] if lines else "(пока пусто)"

def fail(gpu):
    print(f"!!! процесс GPU {gpu} упал, хвост лога:")
    print("\n".join(logs[gpu].read_text(encoding="utf-8", errors="replace").splitlines()[-30:]))
    for p in procs.values():
        p.kill()
    raise RuntimeError(f"обучение на GPU {gpu} упало")

train_end = {}
while len(train_end) < len(procs):
    time.sleep(20 if QUICK_TEST else 60)
    for gpu, p in procs.items():
        timing_file = OUT / f"timing_gpu{gpu}.json"
        if gpu not in train_end and timing_file.exists():
            train_end[gpu] = json.loads(timing_file.read_text(encoding="utf-8"))["t_train_end"]
        elif gpu not in train_end and p.poll() is not None:
            fail(gpu)
        print(last_line(logs[gpu]))

TRAIN_MIN = (max(train_end.values()) - T0) / 60   # ======== КОНЕЦ ТАЙМЕРА ОБУЧЕНИЯ ========
print("=" * 80)
print(f"ВРЕМЯ ОБУЧЕНИЯ ФИНАЛЬНОГО РЕШЕНИЯ: {TRAIN_MIN:.1f} мин (лимит 45 мин) | "
      + " | ".join(f"GPU {g}: {(t - T0) / 60:.1f} мин" for g, t in sorted(train_end.items())))
print("=" * 80)

for gpu, p in procs.items():                       # ждём предсказаний test
    if p.wait() != 0:
        fail(gpu)
timings = {gpu: json.loads((OUT / f"timing_gpu{gpu}.json").read_text(encoding="utf-8")) for gpu in procs}
PREDICT_MIN = (max(t["t_pred_end"] for t in timings.values()) - max(train_end.values())) / 60
print(f"предсказание test после конца обучения: {PREDICT_MIN:.1f} мин (TTA x{TTA_VIEWS})")

**Как читать.** Строки `[guard]` в логах — прогноз времени после первой эпохи каждой модели. Если там «!!!»,
предохранитель уменьшил число эпох: решение уложилось в лимит, но качество может быть чуть ниже. Без «!!!»
все модели прошли полные 20 эпох, как в проверенном рецепте.

## 5. Ансамбль и `final_submission.csv`
Вероятности четырёх моделей усредняются с равными весами. Веса не подбираем: на 699 снимках val это была бы
подгонка под шум. Метка — класс с наибольшей средней вероятностью.

In [ ]:
test_probs = {m: np.load(OUT / f"test_probs_{m}.npy") for m in ALL_MODELS}
ensemble = np.mean(list(test_probs.values()), axis=0)
final = pd.DataFrame({"id": test_table.id, "label": ensemble.argmax(1)})
assert list(final.columns) == ["id", "label"] and len(final) == len(test_table) == (256 if QUICK_TEST else 8299)
assert final.id.tolist() == test_table.id.tolist() and final.label.between(0, 19).all()
FINAL_NAME = "final_submission_quick.csv" if QUICK_TEST else "final_submission.csv"
final.to_csv(FINAL_NAME, index=False)
print(f"{FINAL_NAME}: {len(final)} строк; доли классов "
      f"{final.label.value_counts(normalize=True).min():.1%}...{final.label.value_counts(normalize=True).max():.1%}")
for m, p in test_probs.items():
    print(f"  {m:5s}: совпадает с ансамблем на {(p.argmax(1) == final.label.to_numpy()).mean():.2%} снимков test")

## 6. Итоговая таблица и время
`val acc` — точность holdout-модели с тем же рецептом (обучена на train, проверена на val, TTA ×8) из
`Team_final.ipynb`. Модели в этом ноутбуке обучены на train + val, проверить их на val уже нельзя.

In [ ]:
rows = []
for gpu, t in sorted(timings.items()):
    for m, info in t["per_model"].items():
        rows.append({"модель": m, "GPU": gpu, "эпох": info["epochs"], "обучение, мин": round(info["train_min"], 1),
                     "предсказание, мин": round(info["predict_min"], 1), "val acc (holdout)": HOLDOUT_VAL.get(m),
                     "seed": SEED})
rows.append({"модель": "ансамбль " + " + ".join(ALL_MODELS),
             "val acc (holdout)": HOLDOUT_ENSEMBLE_VAL.get(frozenset(ALL_MODELS))})
report = pd.DataFrame(rows)
report.to_csv("experiments_45min.csv", index=False)
TOTAL_MIN = (time.time() - NOTEBOOK_T0) / 60
summary = {"train_min_wall_clock": round(TRAIN_MIN, 2), "predict_min_after_training": round(PREDICT_MIN, 2),
           "notebook_total_min": round(TOTAL_MIN, 2), "plan": {str(k): v for k, v in PLAN.items()},
           "epochs": EPOCHS, "tta_views": TTA_VIEWS, "versions": VERSIONS}
Path("timing_summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False), encoding="utf-8")
print(report.to_string(index=False))
print(f"\nобучение {TRAIN_MIN:.1f} мин | предсказание {PREDICT_MIN:.1f} мин | весь ноутбук {TOTAL_MIN:.1f} мин")